## 1. Fixed Window
- Use Case: Calculate the average CPU usage per server every 5 minutes.
- Scenario: We have a stream of server CPU usage readings, and we want to compute the average CPU usage every 5 minutes for each server.

In [25]:
import apache_beam as beam
from apache_beam.transforms.window import FixedWindows
from apache_beam.options.pipeline_options import PipelineOptions

# Sample data: (server_id, cpu_usage, timestamp)
cpu_readings = [
    ('server1', 75, '2024-11-14T01:01:30Z'),
    ('server2', 65, '2024-11-14T01:02:00Z'),
    ('server1', 70, '2024-11-14T01:05:30Z'),
    ('server2', 80, '2024-11-14T01:06:15Z'),
    ('server1', 85, '2024-11-14T01:09:45Z'),
    ('server2', 75, '2024-11-14T01:10:00Z'),
]

# Convert timestamp to Beam timestamped value
def add_timestamp(element):
    import datetime
    server_id, cpu, timestamp_str = element
    timestamp = datetime.datetime.fromisoformat(timestamp_str.replace("Z", "+00:00"))
    return beam.window.TimestampedValue((server_id, cpu), timestamp.timestamp())

# Calculate average CPU usage
def calculate_average(data):
    server_id, cpu_list = data
    avg_cpu = sum(cpu_list) / len(cpu_list)
    return server_id, avg_cpu

options = PipelineOptions()
with beam.Pipeline() as p:
    avg_cpu_usage = (
        p
        | "Create CPU Data" >> beam.Create(cpu_readings)
        | "Add Timestamps" >> beam.Map(add_timestamp)
        | "Apply Fixed Window" >> beam.WindowInto(FixedWindows(300))  # 5 minutes window
        | "Group by Server" >> beam.GroupByKey()
        | "Calculate Average CPU" >> beam.Map(calculate_average)
        | "Print Results" >> beam.Map(print)
    )


('server1', 75.0)
('server1', 77.5)
('server2', 65.0)
('server2', 80.0)
('server2', 75.0)


#

## 2. Sliding Window
- Use Case: Track average customer spending in the last 15 minutes, updated every 5 minutes.
- Scenario: We get a stream of purchases, and we want a moving average of customer spending. A sliding window of 15 minutes, sliding every 5 minutes, helps us calculate this moving average.

In [24]:
from apache_beam.transforms.window import SlidingWindows

# Sample data: (customer_id, purchase_amount, timestamp)
purchase_data = [
    ('cust1', 100, '2024-11-14T01:01:00Z'),
    ('cust2', 200, '2024-11-14T01:02:30Z'),
    ('cust1', 50, '2024-11-14T01:08:00Z'),
    ('cust3', 120, '2024-11-14T01:10:00Z'),
    ('cust2', 300, '2024-11-14T01:12:00Z'),
    ('cust1', 90, '2024-11-14T01:14:30Z'),
]

# Convert to timestamped data
def add_timestamp(element):
    import datetime
    customer_id, amount, timestamp_str = element
    timestamp = datetime.datetime.fromisoformat(timestamp_str.replace("Z", "+00:00"))
    return beam.window.TimestampedValue((customer_id, amount), timestamp.timestamp())

# Calculate average spending
def calculate_average(data):
    customer_id, amounts = data
    avg_spending = sum(amounts) / len(amounts)
    return customer_id, avg_spending

with beam.Pipeline() as p:
    avg_spending = (
        p
        | "Create Purchase Data" >> beam.Create(purchase_data)
        | "Add Timestamps" >> beam.Map(add_timestamp)
        | "Apply Sliding Window" >> beam.WindowInto(SlidingWindows(900, 300))  # 15 min window, 5 min slide
        | "Group by Customer" >> beam.GroupByKey()
        | "Calculate Average Spending" >> beam.Map(calculate_average)
        | "Print Results" >> beam.Map(print)
    )


('cust1', 80.0)
('cust1', 75.0)
('cust1', 100.0)
('cust1', 70.0)
('cust1', 90.0)
('cust2', 250.0)
('cust2', 200.0)
('cust2', 200.0)
('cust2', 300.0)
('cust2', 300.0)
('cust3', 120.0)
('cust3', 120.0)
('cust3', 120.0)


#

## 3. Session Window
- Use Case: Calculate the total browsing time for each user in a session.
- Scenario: Each time a user visits the site, a session is created if there’s no activity for 10 minutes or more. We want to group activity within these sessions.

In [23]:
from apache_beam.transforms.window import Sessions

# Sample data: (user_id, browsing_time (seconds), timestamp)
user_activity = [
    ('user1', 30, '2024-11-14T01:00:00Z'),
    ('user2', 20, '2024-11-14T01:02:00Z'),
    ('user1', 40, '2024-11-14T01:03:00Z'),
    ('user1', 50, '2024-11-14T01:15:00Z'),  # New session for user1
    ('user2', 60, '2024-11-14T01:20:00Z'),
]

def add_timestamp(element):
    import datetime
    user_id, duration, timestamp_str = element
    timestamp = datetime.datetime.fromisoformat(timestamp_str.replace("Z", "+00:00"))
    return beam.window.TimestampedValue((user_id, duration), timestamp.timestamp())

# Calculate total browsing time per session
def calculate_total(data):
    user_id, durations = data
    total_time = sum(durations)
    return user_id, total_time

with beam.Pipeline() as p:
    session_durations = (
        p
        | "Create User Activity" >> beam.Create(user_activity)
        | "Add Timestamps" >> beam.Map(add_timestamp)
        | "Apply Session Window" >> beam.WindowInto(Sessions(600))  # 10 min inactivity gap
        | "Group by User" >> beam.GroupByKey()
        | "Calculate Total Browsing Time" >> beam.Map(calculate_total)
        | "Print Results" >> beam.Map(print)
    )


('user1', 70)
('user1', 50)
('user2', 20)
('user2', 60)


#

## 4. Global Window
- Use Case: Calculate the total sales across all regions in a batch job.
- Scenario: For a batch job processing a complete sales dataset, we can use a global window to aggregate all sales without time-based windowing.

In [22]:
from apache_beam.transforms.window import GlobalWindows

# Sample data: (region, sales_amount)
sales_data = [
    ('North', 1000),
    ('South', 1500),
    ('East', 700),
    ('West', 900),
    ('North', 1200),
    ('South', 1300),
]

# Sum sales per region
def sum_sales(data):
    region, sales = data
    total_sales = sum(sales)
    return region, total_sales

with beam.Pipeline() as p:
    total_sales = (
        p
        | "Create Sales Data" >> beam.Create(sales_data)
        | "Apply Global Window" >> beam.WindowInto(GlobalWindows())  # All data in one window
        | "Group by Region" >> beam.GroupByKey()
        | "Sum Sales" >> beam.Map(sum_sales)
        | "Print Results" >> beam.Map(print)
    )


('North', 2200)
('South', 2800)
('East', 700)
('West', 900)
